# JEPA: Systematic Study

Cross-tick latent prediction. Each experiment shows exactly what JEPA params changed.

| group | what changes | runs |
|---|---|---|
| **Main** | +JEPA at w=0.1, 5 seeds x 2 tasks | 10 |
| **Sweep** | weight ∈ {0.02..0.5}, 3 seeds x 2 tasks | 36 |
| **Ablation** | loss/stopgrad/depth/hidden, 3 seeds x 2 tasks | 42 |

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

TASKS = ['sort', 'cifar10', 'mazes', 'parity']
SEEDS_MAIN = [0, 1, 2, 3, 4]
SEEDS_SWEEP = [0, 1, 2]

In [ ]:
for task in ['cifar10', 'mazes']:
    module, cfg = BASE_CONFIGS[task]
    print(f'{task:10s} d_model={cfg.get("d_model")}, iters={cfg.get("iterations")}')

## Prior Results (st04 jepa_w0.1)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    print(summary_stats(df_prior[(df_prior.stage=='st04')&(df_prior.sweep=='jepa_w0.1')]))
    plot_prior_bar(df_prior, ['cifar10','mazes'],
                   'st04', 'jepa_w0.1', 'Prior: JEPA w=0.1', 'figures/02_prior_bar.png')
else:
    print('Prior data not found.')

## Group 1 — Main (10 runs)

**Δ from baseline**:
```
+ cross_tick_jepa_weight        = 0.1    # auxiliary JEPA loss weight
+ cross_tick_jepa_hidden_dim    = 128    # predictor hidden dim
+ cross_tick_jepa_predictor_depth = 2    # predictor MLP depth
+ cross_tick_jepa_dropout       = 0.0    # no dropout in predictor
```

In [ ]:
JEPA_BASE = dict(
    cross_tick_jepa_hidden_dim=128,
    cross_tick_jepa_predictor_depth=2,
    cross_tick_jepa_dropout=0.0,
    cross_tick_jepa_weight=0.1,
)
exps_main = []
for task in ['cifar10', 'mazes']:
    module, base = BASE_CONFIGS[task]
    for s in [0,1,2,3,4]:
        exps_main.append(Experiment(
            name=f'{task}_jepa_w0p1_s{s}',
            task=task, module=module,
            config={**base, 'seed': s, **JEPA_BASE}))
print(f'{len(exps_main)} experiments')

## Group 2 — Weight Sweep (36 runs)

**Only change**: `cross_tick_jepa_weight` ∈ {0.02, 0.05, 0.1, 0.2, 0.3, 0.5}

All other JEPA params fixed at defaults (hidden=128, depth=2, dropout=0).

In [ ]:
SWEEP_WEIGHTS = [0.02, 0.05, 0.1, 0.2, 0.3, 0.5]
exps_sweep = []
for task in ['cifar10', 'mazes']:
    module, base = BASE_CONFIGS[task]
    for w in SWEEP_WEIGHTS:
        for s in [0,1,2]:
            exps_sweep.append(Experiment(
                name=f'{task}_swp_w{str(w).replace(".","p")}_s{s}',
                task=task, module=module,
                config={**base, 'seed': s,
                        'cross_tick_jepa_weight': w,       # SWEEP
                        'cross_tick_jepa_hidden_dim': 128,
                        'cross_tick_jepa_predictor_depth': 2,
                        'cross_tick_jepa_dropout': 0.0}))
print(f'{len(exps_sweep)} experiments  ({len(SWEEP_WEIGHTS)} weights x 3 seeds x 2 tasks)')

## Group 3 — Ablation (42 runs)

| variant | what changes | question |
|---|---|---|
| `full` | nothing (w=0.1 reference) | — |
| `loss_mse` | loss='mse' (default: 'cosine') | Does loss type matter? |
| `no_stopgrad` | target_stop_grad=False | Is stop-grad needed? |
| `depth1` | predictor_depth=1 (default: 2) | Shallower predictor? |
| `depth4` | predictor_depth=4 | Deeper predictor? |
| `hid64` | hidden_dim=64 (default: 128) | Smaller predictor? |
| `hid256` | hidden_dim=256 | Larger predictor? |

In [ ]:
ABLATIONS = [
    ('full',        dict()),
    ('loss_mse',    {'cross_tick_jepa_loss': 'mse'}),
    ('no_stopgrad', {'cross_tick_jepa_target_stop_grad': False}),
    ('depth1',      {'cross_tick_jepa_predictor_depth': 1}),
    ('depth4',      {'cross_tick_jepa_predictor_depth': 4}),
    ('hid64',       {'cross_tick_jepa_hidden_dim': 64}),
    ('hid256',      {'cross_tick_jepa_hidden_dim': 256}),
]
exps_ablation = []
for task in ['cifar10', 'mazes']:
    module, base = BASE_CONFIGS[task]
    for variant, overrides in ABLATIONS:
        for s in [0,1,2]:
            cfg = {**base, 'seed': s, **JEPA_BASE}
            cfg.update(overrides)
            exps_ablation.append(Experiment(
                name=f'{task}_abl_{variant}_s{s}',
                task=task, module=module, config=cfg))
print(f'{len(exps_ablation)} experiments')
for v, _ in ABLATIONS:
    print(f'  {v}')

## Run All

In [ ]:
exps = exps_main + exps_sweep + exps_ablation
print(f'Total: {len(exps)}')
run_all(exps, gpus=8, log_root='logs/deep/02_jepa', dry_run=True)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/02_jepa')

In [ ]:
status('logs/deep/02_jepa')

## Analysis

In [ ]:
df = collect('logs/deep/02_jepa')
if df.empty:
    print('No results yet.')
else:
    df_main = df[df.name.str.contains('jepa_w0p1_s') & ~df.name.str.contains('swp|abl')]
    if not df_main.empty:
        print(df_main[['name','task','best_acc','delta']].to_string(index=False))
        plot_delta_bars(df_main, 'JEPA(0.1) vs baseline', 'figures/02_main_delta.png')
        print(significance_test(df_main).to_string(index=False))

In [ ]:
if not df.empty:
    import re
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        df_sw['weight'] = df_sw['name'].str.extract(r'w([0-9]+p?[0-9]*)_s')[0].str.replace('p','.').astype(float)
        plot_sweep_heatmap(df_sw, 'weight', 'task',
                          'JEPA weight x task delta (pp)', 'figures/02_sweep_heatmap.png')
        plot_sweep_curve(df_sw, 'weight', 'Weight sweep', 'figures/02_sweep_curve.png')

In [ ]:
if not df.empty:
    df_abl = df[df.name.str.contains('abl_')].copy()
    if not df_abl.empty:
        import re
        df_abl['variant'] = df_abl['name'].str.extract(r'abl_([a-z_0-9]+)_s')[0]
        plot_ablation_bars(df_abl, 'variant',
                          'JEPA ablation', 'figures/02_ablation.png')
        print(summary_stats(df_abl, groupby=('task','variant')))